# TIGER / Smartville — Collected-Data Analytics & Open-Set Feasibility

This notebook analyses data captured by **`FlowDataRecorder`** in
`smartville-controller` (`data_collection_mode: true`) and answers one question
in particular: **how feasible is open-set recognition (OSR) on this traffic?**

It is fully self-contained — it does **not** import anything from the controller
codebase. It only reads the on-disk artefacts a capture run produces
(`manifest.json`, `shards_index.jsonl`, `shard_*.pt`) and reconstructs the
open-set curriculum from the manifest exactly as the controller does at replay
time.

---

### What a capture run looks like on disk

```
<data_collection_dir>/
  run_<timestamp>/
    manifest.json        # full config snapshot (curriculum, dims, rewards, ...)
    shards_index.jsonl   # one line per flushed shard (counts, tick range, histogram)
    shard_000000.pt      # torch.save'd dict, see below
    shard_000001.pt
    ...
```

Each `shard_*.pt` is a dict:

| key | dtype / shape | meaning |
|---|---|---|
| `flow_features`   | `float32` `[n, flows_per_sample, flow_feat_dim]` = `[n, 10, 4]` | rolling window of flow-stats vectors |
| `packet_features` | `uint8` `[n, packets_per_sample, packet_feat_dim]` = `[n, 1, 64]` (or `None`) | first 64 raw IP bytes, IP/ports anonymised |
| `node_features`   | `float32` or `None` | node telemetry (off in this run) |
| `element_classes` | `list[str]`, len `n` | **ground-truth** natural-language class |
| `flow_ids`        | `list[str]`, len `n` | `src_dst_port` traceability id |
| `tick`            | `list[int]`, len `n` | arrival-order counter |

**The 4 flow-stat features** (`extract_flow_feature_tensor`): `[byte_count,
duration_nsec / 1e10, duration_sec, packet_count]`.
**The 64 packet features**: the first 64 bytes of the IP packet as integers in
`[0, 255]`, with the IP address bytes (12–19) and transport ports zeroed.

### The open-set curriculum (from `manifest["knowledge"]`)

Only the ground-truth string label is recorded; the `zda` / `test_zda` booleans
are **recomputed here** from the manifest's curriculum, exactly as
`TigerBrain.get_zda_labels` does:

| group | role | OSR meaning |
|---|---|---|
| **Knowns** | closed-set training classes | the classifier's known world — OSR *negatives* |
| **G1s** | training-time pseudo zero-days (`zda=1, test_zda=0`) | novelty seen only by the anomaly detector during training — a **validation** open-set |
| **G2s** | test zero-days (`zda=1, test_zda=1`) | the **true open-set** the system must flag at test time — OSR *positives* |

> **OSR feasibility question:** using only the raw input representation, can a
> detector fit on **Knowns** separate the **G2** open-set from the knowns —
> and which zero-day classes / which feature stream make that easy or hard?


## 0 · Configuration

Point `DATA_ROOT` at your capture directory. It may be either the collection root (containing `run_*/` subdirs — the newest is picked) or a single `run_*/` directory. Everything else has sensible defaults.

In [ ]:
import os

# Collection root (contains run_*/) OR a single run_* dir.
# Overridable via the TIGER_DATA_ROOT env var so the notebook can run headless.
DATA_ROOT = os.environ.get(
    "TIGER_DATA_ROOT",
    "/pox/pox/smartController/tiger_data_collection/",
)
RUN_DIR = os.environ.get("TIGER_RUN_DIR", "") or None   # explicit run_* path, else newest under DATA_ROOT

# offline_replay.py skips the FIRST shard-index entry (an initial packet-repetition
# warm-up artefact of the capture process itself, not real traffic). Mirror that.
SKIP_FIRST_SHARD = True

MAX_SHARDS  = None     # cap number of shards loaded (None = all)
MAX_SAMPLES = None     # cap total samples after loading (None = all)

RANDOM_STATE = 777
SCATTER_MAX  = 6000    # max points drawn in a scatter (balanced subsample)
TSNE_MAX     = 4000    # max points fed to t-SNE (it is O(n^2))
KNOWN_TEST_FRAC = 0.40 # fraction of Known samples held out as OSR negatives

# Validated, colour-blind-safe palette (light surface). Groups read as an
# ordinal novelty ramp: known -> validation-novelty -> open-set.
C = dict(blue="#2a78d6", aqua="#1baf7a", yellow="#eda100", green="#008300",
         violet="#4a3aa7", red="#e34948", magenta="#e87ba4", orange="#eb6834",
         grey="#b8b7b2", ink="#0b0b0b", ink2="#52514e")

GROUP_ORDER  = ["Known (closed-set)", "G1 (train novelty)", "G2 (open-set / test ZdA)", "Other"]
GROUP_COLOR  = {"Known (closed-set)": C["blue"], "G1 (train novelty)": C["yellow"],
                "G2 (open-set / test ZdA)": C["red"], "Other": C["grey"]}
ROLE_COLOR   = {"benign": C["aqua"], "malicious": C["orange"]}
print("DATA_ROOT =", DATA_ROOT)
print("RUN_DIR   =", RUN_DIR or "(newest under DATA_ROOT)")


In [ ]:
%matplotlib inline
import json, glob, warnings
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")
np.random.seed(RANDOM_STATE)

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#cfcfca", "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": "#ececea", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.labelsize": 10.5,
    "axes.titlecolor": C["ink"], "axes.labelcolor": C["ink2"],
    "xtick.color": C["ink2"], "ytick.color": C["ink2"],
    "xtick.labelsize": 9.5, "ytick.labelsize": 9.5,
    "legend.frameon": False, "legend.fontsize": 9.5,
    "font.size": 10.5,
})

def style_ax(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    return ax

print("torch", torch.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)


## 1 · Load the manifest and reconstruct the curriculum

In [ ]:
def resolve_run_dir(data_root, run_dir=None):
    if run_dir:
        if not os.path.isfile(os.path.join(run_dir, "shards_index.jsonl")):
            raise FileNotFoundError(f"{run_dir} has no shards_index.jsonl")
        return run_dir
    if os.path.isfile(os.path.join(data_root, "shards_index.jsonl")):
        return data_root  # data_root is itself a run dir
    cands = [os.path.join(data_root, d) for d in sorted(os.listdir(data_root))]
    cands = [c for c in cands if os.path.isfile(os.path.join(c, "shards_index.jsonl"))]
    if not cands:
        raise FileNotFoundError(
            f"No run_*/ dirs (with shards_index.jsonl) under {data_root}. "
            f"Set DATA_ROOT/RUN_DIR, or run the synthetic smoke-test cell at the bottom.")
    return max(cands, key=os.path.getmtime)  # newest

RUN = resolve_run_dir(DATA_ROOT, RUN_DIR)
with open(os.path.join(RUN, "manifest.json")) as f:
    manifest = json.load(f)

know = manifest["knowledge"]
KNOWNS = list(know["Knowns"])
G1S    = list(know["G1s"])
G2S    = list(know["G2s"])
BENIGN = set(know.get("bening_patterns", []))
ATTACK = set(know.get("attack_patterns", []))
REWARDS = manifest.get("rewards", {})

FLOW_FEAT_NAMES = ["byte_count", "duration_nsec/1e10", "duration_sec", "packet_count"]

def group_of(lbl):
    if lbl in KNOWNS: return "Known (closed-set)"
    if lbl in G1S:    return "G1 (train novelty)"
    if lbl in G2S:    return "G2 (open-set / test ZdA)"
    return "Other"

def role_of(lbl):
    if lbl in BENIGN: return "benign"
    if lbl in ATTACK: return "malicious"
    return "unknown"

print("Run dir :", RUN)
print("Knowns  :", KNOWNS)
print("G1s     :", G1S, " (training pseudo zero-days)")
print("G2s     :", G2S, " (open-set / test zero-days)")
print("benign  :", sorted(BENIGN))
print("malicious:", sorted(ATTACK))


## 2 · Load the shards

In [ ]:
def load_run(run_dir, skip_first=True, max_shards=None, max_samples=None):
    idx_path = os.path.join(run_dir, "shards_index.jsonl")
    index = [json.loads(l) for l in open(idx_path) if l.strip()]
    if skip_first and len(index) > 1:
        index = index[1:]
    if max_shards is not None:
        index = index[:max_shards]

    flow_parts, pkt_parts = [], []
    labels, ticks, flow_ids, shard_of = [], [], [], []
    have_pkt = True
    for si, e in enumerate(index):
        sd = torch.load(os.path.join(run_dir, e["shard"]), map_location="cpu", weights_only=False)
        f = sd["flow_features"]
        flow_parts.append(f.to(torch.float32).numpy())
        p = sd.get("packet_features", None)
        if p is None:
            have_pkt = False
        else:
            pkt_parts.append(p.to(torch.float32).numpy())
        n = f.shape[0]
        labels   += list(sd["element_classes"])
        ticks    += list(sd["tick"])
        flow_ids += list(sd.get("flow_ids", ["?"] * n))
        shard_of += [e["shard"]] * n

    flow = np.concatenate(flow_parts, axis=0)                       # [N, 10, 4]
    pkt  = np.concatenate(pkt_parts, axis=0) if (have_pkt and pkt_parts) else None  # [N, 1, 64]
    labels = np.array(labels)
    ticks  = np.array(ticks)
    if max_samples is not None and len(labels) > max_samples:
        sel = slice(0, max_samples)
        flow, labels, ticks = flow[sel], labels[sel], ticks[sel]
        flow_ids, shard_of = flow_ids[:max_samples], shard_of[:max_samples]
        if pkt is not None: pkt = pkt[sel]
    return flow, pkt, labels, ticks, np.array(flow_ids), np.array(shard_of)

flow, pkt, labels, ticks, flow_ids, shard_of = load_run(
    RUN, SKIP_FIRST_SHARD, MAX_SHARDS, MAX_SAMPLES)

meta = pd.DataFrame({
    "label": labels,
    "group": [group_of(l) for l in labels],
    "role":  [role_of(l)  for l in labels],
    "tick":  ticks,
    "shard": shard_of,
})
N = len(meta)
print(f"Loaded {N:,} samples")
print(f"  flow_features   : {flow.shape}  {flow.dtype}")
print(f"  packet_features : {pkt.shape if pkt is not None else 'None (use_packet_feats was off)'}")
print(f"  distinct classes: {meta['label'].nunique()}   ticks: [{ticks.min()}, {ticks.max()}]")
meta.head()


## 3 · Class balancing

Severe class imbalance is the first thing that makes OSR (and closed-set
training) hard, so we quantify it before anything else. Bars are coloured by
**curriculum group** — the axis that matters for open-set — and counts are
printed directly on each bar (the palette's low-contrast hues require a
non-colour label).

In [ ]:
counts = meta["label"].value_counts()
order  = counts.index.tolist()
grp    = [group_of(l) for l in order]
cols   = [GROUP_COLOR[g] for g in grp]

fig, ax = plt.subplots(figsize=(11, 4.6))
bars = ax.bar(range(len(order)), counts.values, color=cols, width=0.78,
              edgecolor="white", linewidth=0.6)
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=40, ha="right")
ax.set_ylabel("samples"); ax.set_title("Per-class sample counts (coloured by curriculum group)")
for b, v in zip(bars, counts.values):
    ax.text(b.get_x()+b.get_width()/2, v, f"{v:,}", ha="center", va="bottom",
            fontsize=8.3, color=C["ink2"])
handles = [Line2D([0],[0], marker="s", ls="", ms=9, mfc=GROUP_COLOR[g], mec="none",
                  label=g) for g in GROUP_ORDER if g in grp]
ax.legend(handles=handles, loc="upper right", ncol=1)
style_ax(ax); ax.margins(y=0.14); plt.tight_layout(); plt.show()


In [ ]:
# Group- and role-level composition + imbalance metrics.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.9))

g_counts = meta["group"].value_counts().reindex([g for g in GROUP_ORDER if g in set(meta["group"])]).dropna()
axes[0].bar(range(len(g_counts)), g_counts.values,
            color=[GROUP_COLOR[g] for g in g_counts.index], width=0.7, edgecolor="white")
axes[0].set_xticks(range(len(g_counts))); axes[0].set_xticklabels(
    [g.split(" (")[0] for g in g_counts.index], rotation=15, ha="right")
axes[0].set_title("By curriculum group"); axes[0].set_ylabel("samples")
for i, v in enumerate(g_counts.values):
    axes[0].text(i, v, f"{v:,}\n{100*v/N:.0f}%", ha="center", va="bottom", fontsize=8.4, color=C["ink2"])

r_counts = meta["role"].value_counts()
axes[1].bar(range(len(r_counts)), r_counts.values,
            color=[ROLE_COLOR.get(r, C["grey"]) for r in r_counts.index], width=0.6, edgecolor="white")
axes[1].set_xticks(range(len(r_counts))); axes[1].set_xticklabels(r_counts.index)
axes[1].set_title("Benign vs malicious")
for i, v in enumerate(r_counts.values):
    axes[1].text(i, v, f"{v:,}\n{100*v/N:.0f}%", ha="center", va="bottom", fontsize=8.4, color=C["ink2"])

# imbalance metrics
p = counts.values / counts.values.sum()
entropy = -(p * np.log2(p)).sum()
max_entropy = np.log2(len(p))
gini = 1 - (p**2).sum()
imb_ratio = counts.max() / counts.min()
axes[2].axis("off")
txt = (f"classes                 {len(counts)}\n"
       f"total samples           {N:,}\n"
       f"majority / minority     {counts.max():,} / {counts.min():,}\n"
       f"imbalance ratio         {imb_ratio:.1f}x\n"
       f"normalised entropy      {entropy/max_entropy:.3f}  (1=uniform)\n"
       f"Gini impurity           {gini:.3f}\n\n"
       f"most common: {counts.index[0]} ({counts.iloc[0]:,})\n"
       f"rarest:      {counts.index[-1]} ({counts.iloc[-1]:,})")
axes[2].text(0, 1, "Imbalance summary", fontsize=12, weight="bold", va="top", color=C["ink"])
axes[2].text(0, 0.82, txt, fontsize=10, va="top", family="monospace", color=C["ink2"])
for a in axes[:2]: style_ax(a); a.margins(y=0.16)
plt.tight_layout(); plt.show()


In [ ]:
# Temporal composition: does the class mix drift across the capture (ticks)?
nbins = 60
bins = np.linspace(ticks.min(), ticks.max()+1e-9, nbins+1)
bin_idx = np.clip(np.digitize(ticks, bins) - 1, 0, nbins-1)
groups_present = [g for g in GROUP_ORDER if g in set(meta["group"])]
mat = np.zeros((len(groups_present), nbins))
for gi, g in enumerate(groups_present):
    m = (meta["group"].values == g)
    for b in range(nbins):
        mat[gi, b] = np.sum(m & (bin_idx == b))
frac = mat / np.clip(mat.sum(0, keepdims=True), 1, None)

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5.6), sharex=True)
centers = (bins[:-1]+bins[1:])/2
a1.stackplot(centers, mat, labels=groups_present,
             colors=[GROUP_COLOR[g] for g in groups_present], edgecolor="white", linewidth=0.2)
a1.set_ylabel("samples / bin"); a1.set_title("Class-group volume over capture (by tick)")
a1.legend(loc="upper right", ncol=len(groups_present), fontsize=8.5)
a2.stackplot(centers, frac, colors=[GROUP_COLOR[g] for g in groups_present], edgecolor="white", linewidth=0.2)
a2.set_ylabel("fraction / bin"); a2.set_xlabel("tick"); a2.set_ylim(0, 1)
a2.set_title("Group composition (normalised)")
for a in (a1, a2): style_ax(a)
plt.tight_layout(); plt.show()


## 4 · Input-space geometry (PCA)

We build three feature views and standardise each (features live on wildly
different scales — byte counts vs. durations vs. raw bytes):

* **flow-stats** — the `[10, 4]` window flattened to a 40-D vector (`t-9 … t0`);
* **packet-bytes** — the 64 raw IP bytes;
* **combined** — z-scored concatenation of both.

PCA is a linear probe of how much class/novelty structure is *linearly*
present in the raw input — the cheapest possible OSR signal.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Feature matrices
X_flow = flow.reshape(flow.shape[0], -1)                       # [N, 40]
flow_cols = [f"{FLOW_FEAT_NAMES[j]}[t{ -(flow.shape[1]-1-i) }]"
             for i in range(flow.shape[1]) for j in range(flow.shape[2])]
X_pkt  = pkt.reshape(pkt.shape[0], -1) if pkt is not None else None  # [N, 64]

scaler_flow = StandardScaler().fit(X_flow)
Xf = scaler_flow.transform(X_flow)
if X_pkt is not None:
    scaler_pkt = StandardScaler().fit(X_pkt)
    Xp = scaler_pkt.transform(X_pkt)
    X_comb = np.hstack([Xf, Xp])
else:
    Xp, X_comb = None, Xf

VIEWS = {"flow-stats (40-D)": Xf}
if Xp is not None:
    VIEWS["packet-bytes (64-D)"] = Xp
    VIEWS["combined (104-D)"] = X_comb
print("feature views:", {k: v.shape for k, v in VIEWS.items()})

def balanced_subsample(labels_arr, cap, seed=RANDOM_STATE):
    # index subsample that keeps every class visible (up to per-class cap).
    rng = np.random.default_rng(seed)
    idx_by = defaultdict(list)
    for i, l in enumerate(labels_arr): idx_by[l].append(i)
    per = max(1, cap // max(1, len(idx_by)))
    keep = []
    for l, ids in idx_by.items():
        ids = np.array(ids)
        keep.extend(ids if len(ids) <= per else rng.choice(ids, per, replace=False))
    keep = np.array(sorted(keep))
    if len(keep) > cap:
        keep = np.sort(rng.choice(keep, cap, replace=False))
    return keep

sub = balanced_subsample(labels, SCATTER_MAX)
print(f"scatter subsample: {len(sub):,} of {N:,} points")


In [ ]:
def pca_scatter(X, title):
    pca = PCA(n_components=min(10, X.shape[1]), random_state=RANDOM_STATE).fit(X)
    Z = pca.transform(X)
    fig = plt.figure(figsize=(14, 4.4))

    # scree
    ax0 = fig.add_subplot(1, 3, 1)
    ev = pca.explained_variance_ratio_
    ax0.bar(range(1, len(ev)+1), ev*100, color=C["blue"], width=0.7, edgecolor="white")
    ax0.plot(range(1, len(ev)+1), np.cumsum(ev)*100, color=C["orange"], marker="o", ms=4, lw=1.8)
    ax0.set_xlabel("component"); ax0.set_ylabel("% variance")
    ax0.set_title(f"Scree — PC1+PC2 = {ev[:2].sum()*100:.0f}%"); style_ax(ax0)

    zx, zy = Z[sub, 0], Z[sub, 1]
    # coloured by group
    ax1 = fig.add_subplot(1, 3, 2)
    for g in GROUP_ORDER:
        m = (meta["group"].values[sub] == g)
        if m.any():
            ax1.scatter(zx[m], zy[m], s=9, c=GROUP_COLOR[g], alpha=0.55, linewidths=0, label=g.split(" (")[0])
    ax1.set_xlabel("PC1"); ax1.set_ylabel("PC2"); ax1.set_title("by curriculum group")
    ax1.legend(loc="best", markerscale=1.4); style_ax(ax1)

    # coloured by benign/malicious
    ax2 = fig.add_subplot(1, 3, 3)
    for r in ["benign", "malicious", "unknown"]:
        m = (meta["role"].values[sub] == r)
        if m.any():
            ax2.scatter(zx[m], zy[m], s=9, c=ROLE_COLOR.get(r, C["grey"]), alpha=0.55, linewidths=0, label=r)
    ax2.set_xlabel("PC1"); ax2.set_ylabel("PC2"); ax2.set_title("by benign / malicious")
    ax2.legend(loc="best", markerscale=1.4); style_ax(ax2)

    fig.suptitle(f"PCA — {title}", y=1.03, fontsize=13, weight="bold")
    plt.tight_layout(); plt.show()
    return pca

pcas = {name: pca_scatter(X, name) for name, X in VIEWS.items()}


In [ ]:
# Per-class PCA small-multiples (highlight one class at a time against a grey
# backdrop) — the readable way to inspect >8 classes without cycling hues.
Xref = VIEWS["combined (104-D)"] if "combined (104-D)" in VIEWS else Xf
Zref = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Xref)
zx, zy = Zref[sub, 0], Zref[sub, 1]
classes = counts.index.tolist()
ncol = 4; nrow = int(np.ceil(len(classes)/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.1*ncol, 2.7*nrow), sharex=True, sharey=True)
axes = np.array(axes).reshape(-1)
for k, cls in enumerate(classes):
    ax = axes[k]
    ax.scatter(zx, zy, s=5, c="#e3e2df", alpha=0.6, linewidths=0)
    m = (labels[sub] == cls)
    ax.scatter(zx[m], zy[m], s=7, c=GROUP_COLOR[group_of(cls)], alpha=0.8, linewidths=0)
    ax.set_title(f"{cls}  ·  {group_of(cls).split(' (')[0]}", fontsize=9.5)
    ax.tick_params(labelbottom=False, labelleft=False)
    for s in ("top","right","left","bottom"): ax.spines[s].set_visible(False)
for k in range(len(classes), len(axes)): axes[k].axis("off")
fig.suptitle("Per-class location in combined-input PCA space (highlighted vs. all)", y=1.005, fontsize=13, weight="bold")
plt.tight_layout(); plt.show()


## 5 · Structure & separability metrics

Beyond a linear PCA probe: a non-linear **t-SNE** embedding, a **class-centroid
distance** heatmap (which classes overlap in input space), and per-class
**silhouette** scores (how compact/separated each class is — a proxy for how
much class signal the raw representation carries at all).

In [ ]:
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_samples, pairwise_distances

tsub = sub if len(sub) <= TSNE_MAX else np.sort(np.random.default_rng(RANDOM_STATE).choice(sub, TSNE_MAX, replace=False))
Xt = (VIEWS["combined (104-D)"] if "combined (104-D)" in VIEWS else Xf)[tsub]
try:
    Zt = TSNE(n_components=2, init="pca", perplexity=30, learning_rate="auto",
              random_state=RANDOM_STATE).fit_transform(Xt)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
    for g in GROUP_ORDER:
        m = (meta["group"].values[tsub] == g)
        if m.any():
            a1.scatter(Zt[m,0], Zt[m,1], s=10, c=GROUP_COLOR[g], alpha=0.6, linewidths=0, label=g.split(" (")[0])
    a1.set_title("t-SNE (combined input) — by group"); a1.legend(markerscale=1.4)
    for r in ["benign","malicious","unknown"]:
        m = (meta["role"].values[tsub] == r)
        if m.any():
            a2.scatter(Zt[m,0], Zt[m,1], s=10, c=ROLE_COLOR.get(r,C["grey"]), alpha=0.6, linewidths=0, label=r)
    a2.set_title("t-SNE (combined input) — by role"); a2.legend(markerscale=1.4)
    for a in (a1,a2):
        a.set_xticks([]); a.set_yticks([]); style_ax(a)
    plt.tight_layout(); plt.show()
except Exception as ex:
    print("t-SNE skipped:", ex)


In [ ]:
# Class-centroid distances in standardised combined space (row-normalised so
# the nearest OTHER class per row is easy to read). Small off-diagonal = overlap.
Xc = VIEWS["combined (104-D)"] if "combined (104-D)" in VIEWS else Xf
cls_list = counts.index.tolist()
cents = np.vstack([Xc[labels == c].mean(0) for c in cls_list])
D = pairwise_distances(cents)
fig, ax = plt.subplots(figsize=(8.2, 6.8))
im = ax.imshow(D, cmap="Blues")
ax.set_xticks(range(len(cls_list))); ax.set_xticklabels(cls_list, rotation=45, ha="right", fontsize=8.5)
ax.set_yticks(range(len(cls_list)))
ax.set_yticklabels([f"{c}  [{group_of(c).split(' (')[0]}]" for c in cls_list], fontsize=8.5)
# tick colour by group
for t, c in zip(ax.get_yticklabels(), cls_list): t.set_color(GROUP_COLOR[group_of(c)])
for t, c in zip(ax.get_xticklabels(), cls_list): t.set_color(GROUP_COLOR[group_of(c)])
ax.set_title("Euclidean distance between class centroids (standardised combined input)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="distance")
plt.tight_layout(); plt.show()


In [ ]:
# Silhouette per class (how well-separated each class is from the rest).
try:
    ssub = sub
    sil = silhouette_samples(Xc[ssub], labels[ssub])
    per_cls = pd.Series(sil, index=labels[ssub]).groupby(level=0).mean().reindex(cls_list)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(range(len(per_cls)), per_cls.values,
           color=[GROUP_COLOR[group_of(c)] for c in per_cls.index], width=0.72, edgecolor="white")
    ax.axhline(0, color=C["ink2"], lw=0.8)
    ax.set_xticks(range(len(per_cls))); ax.set_xticklabels(per_cls.index, rotation=40, ha="right")
    ax.set_ylabel("mean silhouette"); ax.set_title("Per-class silhouette (higher = more compact & separated)")
    for i, v in enumerate(per_cls.values):
        ax.text(i, v, f"{v:.2f}", ha="center", va="bottom" if v>=0 else "top", fontsize=8, color=C["ink2"])
    style_ax(ax); plt.tight_layout(); plt.show()
    print("overall mean silhouette:", round(float(np.mean(sil)), 3))
except Exception as ex:
    print("silhouette skipped:", ex)


## 6 · Open-set recognition feasibility

**Protocol** (mirrors how the controller actually uses these data):

* **Negatives (known):** samples of the **Knowns** classes — split into a *fit*
  set (defines the "known world") and a held-out *test* set.
* **Positives (open-set):** all **G2** samples — the true test zero-days. The
  detector never sees them while fitting.
* **G1** samples are shown as a *reference* novelty group (what the anomaly
  detector was trained to flag during the curriculum), but are not the target.

We score every test sample with several **unknown-ness** scores fit on the
known *fit* set only, and compute the **AUROC** of separating known-test from
G2 — the headline feasibility number — for each feature view:

| score | definition | mirrors |
|---|---|---|
| `centroid` | min Euclidean distance to a known-class mean | prototype distance |
| `mahalanobis` | min Mahalanobis distance to a known class (Ledoit–Wolf cov.) | `im_models/mahalanobis` variant |
| `kNN` | distance to the *k*-th nearest known-fit point | deep-kNN OSR baseline |
| `MSP` | `1 − max softmax` of a logistic model over knowns | max-softmax baseline |
| `energy` | `−logsumexp(logits)` of that model | `confidence_strategy: energy` |

AUROC 0.5 = no separability (OSR infeasible from that view); 1.0 = perfectly
separable.

In [ ]:
from sklearn.covariance import LedoitWolf
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.special import logsumexp

rng = np.random.default_rng(RANDOM_STATE)

known_mask = meta["group"].values == "Known (closed-set)"
g2_mask    = meta["group"].values == "G2 (open-set / test ZdA)"
g1_mask    = meta["group"].values == "G1 (train novelty)"
known_idx  = np.where(known_mask)[0]

assert known_idx.size > 10, "Too few Known samples to fit an OSR detector."
assert g2_mask.sum() > 0, "No G2 (open-set) samples present — cannot measure OSR feasibility."

perm = rng.permutation(known_idx)
n_test = int(len(perm) * KNOWN_TEST_FRAC)
known_test_idx = perm[:n_test]
known_fit_idx  = perm[n_test:]
g2_idx = np.where(g2_mask)[0]
g1_idx = np.where(g1_mask)[0]

y_known_labels = labels[known_fit_idx]
known_classes = sorted(map(str, set(y_known_labels)))
print(f"known fit: {len(known_fit_idx):,} | known test: {len(known_test_idx):,} | "
      f"G2 open-set: {len(g2_idx):,} | G1 ref: {len(g1_idx):,}")
print("known classes used to fit:", known_classes)

def osr_scores(X):
    Xfit = X[known_fit_idx]
    # class means + Ledoit-Wolf (shrunken, invertible) precision per class
    means, precs = {}, {}
    for c in known_classes:
        Xc_ = Xfit[y_known_labels == c]
        means[c] = Xc_.mean(0)
        if len(Xc_) > 1:
            precs[c] = LedoitWolf().fit(Xc_).precision_
        else:
            precs[c] = np.eye(X.shape[1])
    def centroid(Z):
        return np.min(np.stack([np.linalg.norm(Z - means[c], axis=1) for c in known_classes], 1), 1)
    def maha(Z):
        ds = []
        for c in known_classes:
            d = Z - means[c]
            ds.append(np.sqrt(np.maximum(np.einsum("ij,jk,ik->i", d, precs[c], d), 0)))
        return np.min(np.stack(ds, 1), 1)
    knn = NearestNeighbors(n_neighbors=min(5, len(Xfit))).fit(Xfit)
    def knn_score(Z):
        return knn.kneighbors(Z)[0][:, -1]
    scores = {"centroid": centroid, "mahalanobis": maha, "kNN": knn_score}
    # softmax model (needs >=2 known classes)
    if len(known_classes) >= 2:
        clf = LogisticRegression(max_iter=2000).fit(Xfit, y_known_labels)
        def msp(Z):   return 1.0 - clf.predict_proba(Z).max(1)
        def energy(Z):return -logsumexp(clf.decision_function(Z), axis=1)
        scores["MSP"] = msp
        scores["energy"] = energy
    return scores

# y=1 for open-set (G2), y=0 for known-test
eval_idx = np.concatenate([known_test_idx, g2_idx])
y_true   = np.concatenate([np.zeros(len(known_test_idx)), np.ones(len(g2_idx))])

auroc = {}          # view -> {score: auroc}
score_cache = {}    # (view) -> dict of arrays over eval_idx  (+ g1)
for vname, X in VIEWS.items():
    sc = osr_scores(X)
    auroc[vname] = {}
    score_cache[vname] = {}
    for sname, fn in sc.items():
        s_eval = fn(X[eval_idx])
        auroc[vname][sname] = roc_auc_score(y_true, s_eval)
        score_cache[vname][sname] = dict(eval=s_eval,
                                         g1=fn(X[g1_idx]) if len(g1_idx) else np.array([]))
auroc_df = pd.DataFrame(auroc).T
print("\nAUROC (known-test vs G2 open-set):")
auroc_df.round(3)


In [ ]:
# AUROC heatmap across feature views x scores.
fig, ax = plt.subplots(figsize=(1.6+1.2*auroc_df.shape[1], 1.2+0.7*auroc_df.shape[0]))
im = ax.imshow(auroc_df.values, cmap="Blues", vmin=0.5, vmax=1.0, aspect="auto")
ax.set_xticks(range(auroc_df.shape[1])); ax.set_xticklabels(auroc_df.columns)
ax.set_yticks(range(auroc_df.shape[0])); ax.set_yticklabels(auroc_df.index)
for i in range(auroc_df.shape[0]):
    for j in range(auroc_df.shape[1]):
        v = auroc_df.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                color="white" if v > 0.8 else C["ink"], fontsize=10, weight="bold")
ax.set_title("Open-set AUROC  (0.5 = infeasible, 1.0 = perfectly separable)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="AUROC")
plt.tight_layout(); plt.show()

# best (view, score)
best_view, best_score = auroc_df.stack().idxmax()
best_auroc = auroc_df.loc[best_view, best_score]
print(f"Best: {best_score} on {best_view}  ->  AUROC {best_auroc:.3f}")


In [ ]:
# Score distributions (known-test vs G1-ref vs G2-open-set) + ROC, for the best view.
best = score_cache[best_view][best_score]
s_known = best["eval"][:len(known_test_idx)]
s_g2    = best["eval"][len(known_test_idx):]
s_g1    = best["g1"]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.4))
bins = np.linspace(np.percentile(np.concatenate([s_known, s_g2]), 0.5),
                   np.percentile(np.concatenate([s_known, s_g2]), 99.5), 45)
a1.hist(s_known, bins=bins, color=C["blue"], alpha=0.65, density=True, label="Known (test)")
if len(s_g1): a1.hist(s_g1, bins=bins, color=C["yellow"], alpha=0.55, density=True, label="G1 (ref novelty)")
a1.hist(s_g2, bins=bins, color=C["red"], alpha=0.6, density=True, label="G2 (open-set)")
a1.set_xlabel(f"{best_score} unknown-ness  ·  {best_view}"); a1.set_ylabel("density")
a1.set_title("Score distributions — separation = feasibility"); a1.legend(); style_ax(a1)

fpr, tpr, _ = roc_curve(y_true, best["eval"])
a2.plot(fpr, tpr, color=C["blue"], lw=2.2, label=f"G2 vs Known  (AUROC {best_auroc:.3f})")
a2.plot([0,1],[0,1], color=C["grey"], ls="--", lw=1.2, label="chance")
a2.set_xlabel("false positive rate (knowns flagged)"); a2.set_ylabel("true positive rate (G2 caught)")
a2.set_title(f"Open-set ROC — {best_score} on {best_view}"); a2.legend(loc="lower right"); style_ax(a2)
plt.tight_layout(); plt.show()


In [ ]:
# Per-open-set-class detectability: AUROC of each G2 class vs known-test, using
# the best (view, score). Which zero-days are easy vs hard to flag?
Xbest = VIEWS[best_view]
sc = osr_scores(Xbest)[best_score]
s_known_test = sc(Xbest[known_test_idx])
rows = []
for c in [g for g in G2S if (labels == g).any()]:
    cidx = np.where(labels == c)[0]
    s_c = sc(Xbest[cidx])
    yy = np.concatenate([np.zeros(len(s_known_test)), np.ones(len(s_c))])
    ss = np.concatenate([s_known_test, s_c])
    rows.append((c, roc_auc_score(yy, ss), len(cidx)))
det = pd.DataFrame(rows, columns=["class", "auroc", "n"]).sort_values("auroc")

fig, ax = plt.subplots(figsize=(10.5, 0.5+0.5*len(det)))
ax.barh(range(len(det)), det["auroc"].values, color=C["red"], alpha=0.85, edgecolor="white", height=0.7)
ax.axvline(0.5, color=C["ink2"], ls="--", lw=1)
ax.set_yticks(range(len(det))); ax.set_yticklabels([f"{c}  (n={n})" for c, n in zip(det["class"], det["n"])])
ax.set_xlabel("AUROC vs known-test"); ax.set_xlim(0.4, 1.0)
ax.set_title(f"Per-zero-day detectability — {best_score} on {best_view}")
for i, v in enumerate(det["auroc"].values):
    ax.text(v, i, f" {v:.2f}", va="center", ha="left", fontsize=9, color=C["ink2"])
style_ax(ax); plt.tight_layout(); plt.show()
det.round(3)


In [ ]:
# Operating-point view: as we raise the unknown-ness threshold, how many G2 do
# we catch (recall) vs how many knowns we wrongly flag? Mark 90/95% known-retention.
fpr, tpr, thr = roc_curve(y_true, score_cache[best_view][best_score]["eval"])
known_retention = 1 - fpr

def recall_at_retention(fpr, tpr, target):
    # best open-set recall achievable while keeping >= target of knowns (fpr <= 1-target).
    mask = fpr <= (1 - target) + 1e-9
    return float(tpr[mask].max()) if mask.any() else 0.0

fig, ax = plt.subplots(figsize=(8.4, 5))
ax.plot(known_retention, tpr, color=C["blue"], lw=2.2)
# Markers on the curve + a fixed, collision-free label block (the two operating
# points can coincide when separability is high, so labels are anchored, not arrowed).
for ty, (target, col) in enumerate([(0.95, C["orange"]), (0.90, C["violet"])]):
    rec = recall_at_retention(fpr, tpr, target)
    ax.scatter([target], [rec], color=col, zorder=5, s=45)
    ax.text(0.60, 0.22 - 0.08*ty, f"{int(target*100)}% knowns kept  ->  {rec*100:.0f}% G2 caught",
            transform=ax.transAxes, color=col, fontsize=9.5, weight="bold")
ax.set_xlabel("known-traffic retention  (1 - FPR)"); ax.set_ylabel("open-set recall  (G2 TPR)")
ax.set_title(f"OSR operating trade-off — {best_score} on {best_view}")
ax.set_xlim(0.5, 1.005); ax.set_ylim(0, 1.02); style_ax(ax); plt.tight_layout(); plt.show()


## 7 · Automated verdict

In [ ]:
def verdict(a):
    if a >= 0.90: return "STRONG — open-set classes are highly separable from the raw input alone."
    if a >= 0.80: return "GOOD — clear separation; a lightweight OSR head should work well."
    if a >= 0.70: return "MODERATE — usable signal, but expect confusion on the hardest zero-days."
    if a >= 0.60: return "WEAK — limited linear/geometric signal; richer (learned) features likely needed."
    return "POOR — near chance; the raw input barely distinguishes these zero-days from knowns."

hard = det.iloc[0]; easy = det.iloc[-1]
best_stream = auroc_df.mean(1).idxmax()
print("="*74)
print("OPEN-SET RECOGNITION FEASIBILITY — SUMMARY")
print("="*74)
print(f"Samples analysed         : {N:,}   ({meta['label'].nunique()} classes)")
print(f"Class imbalance ratio    : {counts.max()/counts.min():.1f}x  (majority {counts.index[0]} / rarest {counts.index[-1]})")
print(f"Known (closed-set)       : {KNOWNS}")
print(f"G2 open-set (test ZdA)   : {[g for g in G2S if (labels==g).any()]}")
print("-"*74)
print(f"Best detector            : '{best_score}' on '{best_view}'  ->  AUROC {best_auroc:.3f}")
print(f"Most informative stream  : {best_stream}  (mean AUROC {auroc_df.mean(1)[best_stream]:.3f})")
print(f"Easiest zero-day to flag : {easy['class']}  (AUROC {easy['auroc']:.3f})")
print(f"Hardest zero-day to flag : {hard['class']}  (AUROC {hard['auroc']:.3f})")
_fpr, _tpr, _ = roc_curve(y_true, score_cache[best_view][best_score]['eval'])
_m = _fpr <= 0.05 + 1e-9
tpr95 = float(_tpr[_m].max()) if _m.any() else 0.0
print(f"At 95% known-retention   : {tpr95*100:.0f}% of G2 open-set caught")
print("-"*74)
print("VERDICT:", verdict(best_auroc))
print("="*74)


### Caveats & how this maps to the real system

* This is a **feasibility probe on the raw recorded input** (flow-stats window +
  raw packet bytes). The deployed IM learns a representation on top of these,
  trained with an anomaly-detection loss whose positives are the **G1** classes —
  so the production OSR head can do **better** than these linear/geometric
  scores where a class is non-linearly separable, and it is explicitly tuned on
  the G1 novelty that here we only use as a reference.
* AUROC uses **ground-truth** labels; it measures *how much open-set signal
  exists in the data*, not the accuracy of any particular deployed detector.
* The flow window is zero-pre-filled, so early-in-a-flow samples carry partial
  windows; that is real captured behaviour and is kept as-is.
* Rerun with a different `RUN_DIR`, toggle `SKIP_FIRST_SHARD`, or raise
  `MAX_SAMPLES` to check stability across captures.


## Appendix · Synthetic smoke-test data (optional)

No real capture handy? Run the cell below to write a tiny run in the **exact**
`FlowDataRecorder` format to `./_synth_smoketest/`, then set
`DATA_ROOT = "./_synth_smoketest"` at the top and *Run All*. It reads
`Knowns/G1s/G2s` from this run's `manifest` if available, else uses the paper
defaults, so the whole notebook exercises without touching real data.
The values are random — for pipeline validation only, not analysis.

In [ ]:
def write_synthetic_smoketest(out="./_synth_smoketest", n_total=3000, seed=RANDOM_STATE):
    import time, math
    r = np.random.default_rng(seed)
    try:
        km = manifest["knowledge"]; base_manifest = manifest
    except NameError:
        km = {"bening_patterns": ["echo","doorlock","hue"],
              "attack_patterns": ["hakai","mirai","hajime","gafgyt","muhstik","h_scan",
                                   "okiru","generic_ddos","torii","cc_heartbeat"],
              "Knowns": ["hue","hakai","torii"],
              "G1s": ["okiru","cc_heartbeat","generic_ddos"],
              "G2s": ["mirai","gafgyt","hajime","h_scan","muhstik","doorlock","echo"]}
        base_manifest = {"knowledge": km, "flow_feat_dim": 4, "packet_feat_dim": 64,
                         "use_packet_feats": True, "use_node_feats": False,
                         "intrusion_detection": {"flows_per_sample": 10, "packets_per_sample": 1}}
    allc = km["bening_patterns"] + km["attack_patterns"]
    benign, attack, knowns, g1s, g2s = (set(km["bening_patterns"]), set(km["attack_patterns"]),
                                        km["Knowns"], km["G1s"], km["G2s"])
    run = os.path.join(out, "run_" + time.strftime("%Y%m%d_%H%M%S"))
    os.makedirs(run, exist_ok=True)
    json.dump(base_manifest, open(os.path.join(run, "manifest.json"), "w"), indent=2, default=str)
    rows = []
    for i, c in enumerate(allc):
        n = max(15, int(n_total/len(allc) * (2.2 if c in knowns else (0.6 if c in g2s else 1.0))))
        ang = i/len(allc)*2*math.pi; mal = c in attack; off = 0 if c in knowns else (60 if c in g2s else 30)
        fmean = np.array([(5e4 if mal else 8e3)*(1+0.6*math.cos(ang)), 0.4+0.3*math.sin(ang),
                          2+1.5*math.cos(ang), (120 if mal else 25)*(1+0.5*math.sin(ang))])
        tmpl = np.zeros(64); tmpl[0]=69; tmpl[9]=6 if mal else 17
        tmpl = np.clip(tmpl + off*math.sin(ang) + r.integers(0,120,64), 0, 255); tmpl[12:20]=0
        for _ in range(n):
            w = np.zeros((10,4)); s0 = r.integers(0,10)
            for t in range(s0,10): w[t] = fmean*((t-s0+1)/(10-s0))*r.normal(1,0.15,4)
            pk = np.clip(tmpl + r.normal(0,10,64),0,255); pk[12:20]=0
            rows.append((c, np.clip(w,0,None).astype(np.float32), pk.astype(np.uint8)))
    r.shuffle(rows)
    flow_t = torch.from_numpy(np.stack([x[1] for x in rows]))
    pkt_t  = torch.from_numpy(np.stack([x[2] for x in rows])).unsqueeze(1).to(torch.uint8)
    ticks_ = np.repeat(np.arange((len(rows)+4)//5), 5)[:len(rows)]
    torch.save({"flow_features": flow_t, "packet_features": pkt_t, "node_features": None,
                "element_classes": [x[0] for x in rows], "flow_ids": ["synthetic"]*len(rows),
                "tick": ticks_.tolist()}, os.path.join(run, "shard_000000.pt"))
    open(os.path.join(run, "shards_index.jsonl"), "w").write(json.dumps({
        "shard":"shard_000000.pt","num_samples":len(rows),
        "tick_min":int(ticks_.min()),"tick_max":int(ticks_.max())})+"\n")
    print("wrote", run, "->  set DATA_ROOT =", repr(out), "and Run All")
    return out

# write_synthetic_smoketest()   # <- uncomment to generate
